# 使用 OpenAI 的网站摘要器

一个有趣的网络爬虫，可以获取任何网站并使用 OpenAI 的 API 生成尖刻、幽默的摘要。

## 这是做什么的

1. 从 URL 获取网站内容
2.发送到OpenAI的GPT模型
3.返回markdown格式的尖刻摘要

非常适合快速了解网站，而无需阅读所有废话！

## 设置

确保您的 .env 文件位于存储库根目录中：
```
OPENAI_API_KEY=your_key_here
```

然后按顺序运行下面的单元格。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 进口

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

## 连接到 OpenAI

In [ ]:
# 加载环境变量
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# 验证密钥
if not api_key:
    print("Error: No API key found in .env")
elif not api_key.startswith("sk-proj-"):
    print("Warning: API key doesn't look like a valid OpenAI key")
else:
    print("API key loaded successfully!")

## 定义提示

系统提示告诉 GPT 如何表现。
用户提示就是我们要求它执行的操作。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 系统提示-定义AI的性格和角色
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 用户提示-实际请求
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## 构建消息结构

OpenAI 需要具有特定格式和角色（系统/用户/助理）的消息

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def messages_for(website):
    """Create the message structure for OpenAI API"""
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

## 创建摘要函数

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def summarize(url):
    """Fetch a website and return a snarky summary"""
    openai = OpenAI()
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages_for(website)
    )
    return response.choices[0].message.content

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def display_summary(url):
    """Display the summary as formatted markdown"""
    summary = summarize(url)
    display(Markdown(summary))

## 尝试一下！

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 尝试一个简单的网站
display_summary("https://edwarddonner.com")

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 尝试更多网站
display_summary("https://example.com")

## 注释

- 这最适合静态 HTML 网站
- Javascript 渲染的网站（React 应用程序、SPA）不会显示其内容
- 某些具有CDN保护的网站可能会返回403错误
- 对于动态站点，请查看社区文件夹中的 Selenium 或 Playwright 实现

## 定制

想改变语气吗？将 `system_prompt` 变量编辑为：
- 删除专业摘要中的“尖酸刻薄”
- 添加翻译语言说明
- 调整不同的语气（正式、技术、有趣等）